import all libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType

Find the external location

In [0]:
%sql
describe external location `control_datalake_processed`

Extract the destination URL

In [0]:
processed_dir = spark.sql("describe external location `control_datalake_processed`").select("url").collect()[0][0]
print(processed_dir)

Load the raw table information

In [0]:
raw_synthetic = spark.read.table("raw_control_datalake.dev.raw_synthetic_data")
raw_data_overview = spark.read.table("raw_control_datalake.dev.raw_data_overview")
raw_data_dictionary = spark.read.table("raw_control_datalake.dev.raw_data_dictionary")
raw_data_anomaly = spark.read.table("raw_control_datalake.dev.raw_data_anomaly")

implement the required transformations for silver

In [0]:
raw_synthetic_df = raw_synthetic.drop("id")
raw_synthetic_df = raw_synthetic_df.withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_overview_df = raw_data_overview.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_dictionary_df = raw_data_dictionary.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_anomaly_df = raw_data_anomaly.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))

In [0]:
catalog = "process_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")
raw_synthetic_df.write.mode("overwrite").saveAsTable("processed_synthetic_data")
raw_data_overview_df.write.mode("overwrite").saveAsTable("processed_data_overview")
raw_data_dictionary_df.write.mode("overwrite").saveAsTable("processed_data_dictionary")
raw_data_anomaly_df.write.mode("overwrite").saveAsTable("processed_data_anomaly")